In [1]:
"""
PIPELINE STAGE: Generation Dataset Cleaning & Structural Refinement

INPUT: processed_data/steps/04_Generation.xlsx
OUTPUT: processed_data/steps/05_Generation_Cleaned.xlsx
LIBRARIES: os, pandas

1. OBJECTIVE:
    Clean the raw renewable electricity generation dataset extracted from monthly
    energy reports. The workflow removes unnecessary energy-source columns,
    eliminates aggregate summary records, standardizes province names, and
    prepares the dataset for integration with installed-capacity, demographic,
    meteorological, and environmental datasets.

2. COLUMN FILTERING & DATASET REFINEMENT:
    A. Column Name Normalization:
       - Remove leading and trailing spaces.
       - Convert column names to lowercase.
       - Normalize Turkish character encoding artifacts:

           replace("i̇", "i")

    B. Unwanted Generation Sources:
       Remove the following columns whenever present:

           * doğal gaz
           * linyit
           * toplam

       These variables are excluded to retain only renewable-energy
       generation indicators for subsequent analysis.

    C. Dynamic Detection:
       - Search columns programmatically.
       - Remove target columns regardless of formatting differences.

3. PROVINCE COLUMN IDENTIFICATION:
    A. Flexible Detection:
       Locate the province column using keyword matching:

           "iller"

       after text normalization.

    B. Validation:
       - Continue province-level processing only if a valid
         province column is detected.
       - Generate a warning if the column cannot be found.

4. SPATIAL STANDARDIZATION:
    A. Province Name Cleaning:
       - Remove unnecessary whitespace.
       - Convert all province names to uppercase.

    B. Turkish-Aware Capitalization:
       Preserve Turkish language rules during conversion.

       Examples:

           istanbul → İSTANBUL
           izmir    → İZMİR
           çorum    → ÇORUM

    C. Geographic Consistency:
       Ensure all province names follow a standardized format
       suitable for relational joins and spatial analysis.

5. SUMMARY RECORD REMOVAL:
    A. Aggregate Record Detection:
       Identify rows where the province field equals:

           GENEL TOPLAM

    B. Dataset Filtering:
       Remove all summary rows from the dataset.

    C. Analytical Integrity:
       Retain only province-level electricity generation records.

6. DATA QUALITY CONTROL:
    A. Province Column Validation:
       Verify successful detection of the province identifier column.

    B. Record Validation:
       Compare dataset size before and after filtering.

    C. Quality Reporting:
       Report the number of deleted aggregate rows.

7. OUTPUT GENERATION:
    A. Export Clean Dataset:

           processed_data/steps/05_Generation_Cleaned.xlsx

    B. Export Settings:
       - Excel format (.xlsx)
       - No index column

8. PROCESS LOGGING & MONITORING:
    A. Runtime Messages:
       Display:

           - Source file name
           - Removed columns
           - Province standardization status
           - Number of removed summary rows

    B. Warning Messages:
       Report:
           - Missing province column
           - Structural inconsistencies

    C. Completion Messages:
       Confirm successful creation and save location of the cleaned dataset.

9. EXPECTED OUTPUT DATASET:
    The resulting dataset contains:

        - Province-level renewable electricity generation values
        - Standardized uppercase province names
        - Year
        - Month
        - Renewable energy generation indicators only

    Fossil-fuel generation variables and aggregate summary records
    are removed, producing a clean and analysis-ready renewable
    electricity generation dataset for downstream statistical,
    environmental, and machine learning workflows.
"""

import os
import pandas as pd

# 1. Dosya Yolları
BASE_DIR = r"C:\Users\w11\dergi2"
input_file = os.path.join(BASE_DIR, "processed_data", "steps", "04_Generation.xlsx")

# Temizlenmiş veriyi aynı isimle üzerine yazabilir veya yeni isimle kaydedebiliriz. 
# Güvenlik için "_cleaned" ekiyle kaydedelim, istersen değiştirebilirsin.
output_file = os.path.join(BASE_DIR, "processed_data", "steps", "05_generation_cleaned.xlsx")

def clean_capacity_data(input_path, output_path):
    print(f">>> Veri okunuyor: {os.path.basename(input_path)}")
    
    # Excel dosyasını oku
    df = pd.read_excel(input_path)
    
    # --- 1, 2, 3: SÜTUN SİLME İŞLEMİ ---
    # Sütun isimlerinde Word'den kalan gizli boşluklar olabilir diye isimleri temizleyip kontrol ediyoruz.
    cols_to_remove = ["doğal gaz", "linyit", "toplam"]
    
    for col in list(df.columns):
        clean_col_name = str(col).strip().lower().replace("i̇", "i")
        if clean_col_name in cols_to_remove:
            df.drop(columns=[col], inplace=True)
            print(f" [✓] Sütun silindi: {col}")

    # --- 4 & 5: İLLER SÜTUNU İŞLEMLERİ (Satır Silme ve Büyük Harf) ---
    # Öncelikle sütun isminin gerçekten "İLLER" veya benzeri olduğundan emin olalım
    iller_col = None
    for col in df.columns:
        if "iller" in str(col).strip().lower().replace("i̇", "i"):
            iller_col = col
            break
            
    if iller_col:
        # Türkçe karakterleri koruyarak büyük harfe çeviren özel fonksiyon
        def turkce_buyuk_harf(metin):
            if pd.isna(metin):
                return metin
            metin = str(metin).strip()
            return metin.replace("i", "İ").replace("ı", "I").upper()
            
        # 5. Adım: Tüm illeri (ve satırları) Türkçe kurallarına göre büyük harf yap
        df[iller_col] = df[iller_col].apply(turkce_buyuk_harf)
        print(" [✓] Bütün iller büyük harfe çevrildi.")

        # 4. Adım: "GENEL TOPLAM" satırını sil
        # Artık hepsi büyük harf olduğu için bulması çok kolay
        before_count = len(df)
        df = df[df[iller_col] != "GENEL TOPLAM"]
        after_count = len(df)
        
        if before_count != after_count:
            print(f" [✓] 'Genel Toplam' satırı silindi (Silinen satır sayısı: {before_count - after_count})")
    else:
        print(" [!] Hata: 'İLLER' sütunu bulunamadı!")

    # Temizlenmiş veriyi kaydet
    df.to_excel(output_path, index=False)
    print(f"\n✅ TEMİZLİK TAMAMLANDI! Dosya kaydedildi: {output_path}")


# --- KODU ÇALIŞTIR ---
if __name__ == "__main__":
    # Eğer dosya adın xlsx değil de csv ise yukarıdaki yolları .csv olarak değiştirmeyi unutma.
    if os.path.exists(input_file):
        clean_capacity_data(input_file, output_file)
    else:
        print(f"Hata: İşlenecek dosya bulunamadı -> {input_file}")

>>> Veri okunuyor: 04_Generation.xlsx
 [✓] Sütun silindi: Toplam
 [✓] Sütun silindi: Doğal Gaz
 [✓] Bütün iller büyük harfe çevrildi.
 [✓] 'Genel Toplam' satırı silindi (Silinen satır sayısı: 60)

✅ TEMİZLİK TAMAMLANDI! Dosya kaydedildi: C:\Users\w11\dergi2\processed_data\steps\05_generation_cleaned.xlsx
